# Is `stress` its own quantity — or KL / density / activation-norm in disguise?

A one-question notebook. It does two jobs at once:

1. **Teaches the notation by making it run.** Every symbol in the stress
   formula becomes an array you can print and poke, so you can read and
   defend the definition instead of squinting at it.
2. **Produces a real result.** It tests whether `stress` carries information
   the metrics you already log don't. *"Stress is density restated"* is a
   finding. *"Stress carries structure the named metrics miss"* is a finding.
   Either way you leave with something publishable and defensible.

Run it top to bottom. It works out of the box on a tiny embedded sample
(smoke test only — numbers meaningless), then you point `EXPORT_PATH` at your
full session export for the real answer.

## What `stress` actually is

Straight from `analyzer.py::_extract_stress_score`. For token $t$:

$$\mathrm{stress}(t)\;=\;\frac{1}{L}\sum_{\ell\in\text{layers}}\;\sum_{p\in\{q,k,v\}}\frac{\bigl\lVert\,h_t^{(\ell)}\,\Delta W_p^{(\ell)\top}\,\bigr\rVert_2}{\bigl\lVert\,\Delta W_p^{(\ell)}\,\bigr\rVert_F}\qquad\qquad \mathrm{stress\_score}=\frac{1}{T}\sum_{t=1}^{T}\mathrm{stress}(t)$$

Symbol by symbol:

- $h_t^{(\ell)}\in\mathbb{R}^{d}$ — the residual-stream activation at layer $\ell$, token $t$.
- $\Delta W_p^{(\ell)} = W_p^{\text{instruct}}-W_p^{\text{base}}$ — the **weight delta**: how alignment fine-tuning changed the $q$/$k$/$v$ projection at layer $\ell$. This is the uncommon move — you run live activations through the *fine-tuning update itself*.
- $\lVert\cdot\rVert_2$ — vector L2 norm. $\lVert\cdot\rVert_F$ — matrix Frobenius norm ($\sqrt{\sum \sigma_i^2}$).
- $L$ = number of layers measured, $T$ = number of tokens.

**In one sentence:** stress measures how strongly a token's activation excites the directions fine-tuning reshaped in the attention projections. Where alignment changed the model most, and this token's representation lives there, stress is high.

**Trivial cases (your sanity checks):**
- $h_t$ orthogonal to $\Delta W_p$'s row space $\Rightarrow$ that term is $0$: the token doesn't touch the changed subspace.
- $h_t$ aligned with $\Delta W_p$'s top singular direction $\Rightarrow$ that term is maximal.
- $\Delta W_p=0$ (layer/role untouched by tuning) $\Rightarrow$ skipped (the code guards $\lVert\Delta W_p\rVert_F>0$).

**What it is *not*:** KL and entropy are output-distribution quantities; stress is activation-space, mediated by the weight delta. Its nearest cousin is your **SFD density** (proximity of $h$ to the alignment-delta subspace) — which is exactly why the test below pits stress against density first.

**One honest caveat baked into the units:** $\Delta W_p$ sits in both the numerator (as operator) and denominator (as norm), so stress scales roughly like $\lVert h_t\rVert$. That makes *"is stress just tracking activation magnitude?"* a real confound — flagged in Part 2.

In [ ]:
import json, numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt

# ── Point this at your export (the per-record JSONL you pasted, or a .json array) ──
EXPORT_PATH = ""   # e.g. "/path/to/session_export.jsonl"  — leave "" to use the demo sample
METRICS = ["stress_score", "kl_divergence", "density_mean"]

In [ ]:
# 8 records from your export — smoke-test only (4 distinct points; numbers are NOT a result)
DEMO = [
 dict(index=0, category="benign",             n_tokens=9,  stress_score=3.2041475772857666, kl_divergence=0.29561880230903625, density_mean=0.4018205171852309),
 dict(index=1, category="benign:response",    n_tokens=64, stress_score=3.529582977294922,  kl_divergence=0.020691419020295143, density_mean=0.3540464540991512),
 dict(index=4, category="jailbreak",          n_tokens=9,  stress_score=3.2353570461273193, kl_divergence=0.5289775729179382,  density_mean=0.4101165317716389),
 dict(index=5, category="jailbreak:response", n_tokens=32, stress_score=3.926872491836548,  kl_divergence=4.912341594696045,   density_mean=0.3943210538606457),
 dict(index=6, category="jailbreak",          n_tokens=9,  stress_score=3.2353570461273193, kl_divergence=0.5289775729179382,  density_mean=0.4101165317716389),
 dict(index=7, category="jailbreak:response", n_tokens=32, stress_score=3.926872491836548,  kl_divergence=4.912341594696045,   density_mean=0.3943210538606457),
 dict(index=8, category="jailbreak",          n_tokens=9,  stress_score=3.2353570461273193, kl_divergence=0.5289775729179382,  density_mean=0.4101165317716389),
 dict(index=9, category="jailbreak:response", n_tokens=32, stress_score=3.926872491836548,  kl_divergence=4.912341594696045,   density_mean=0.3943210538606457),
]

def load_records(path):
    """Read a JSONL (one object per line) or a JSON array. Keep records that have stress_score."""
    with open(path) as f:
        raw = f.read().strip()
    recs = []
    try:                                   # try JSON array first
        obj = json.loads(raw)
        recs = obj if isinstance(obj, list) else [obj]
    except json.JSONDecodeError:           # fall back to JSONL
        for line in raw.splitlines():
            line = line.strip()
            if line:
                recs.append(json.loads(line))
    return [r for r in recs if r.get("stress_score") is not None]

records = load_records(EXPORT_PATH) if EXPORT_PATH else DEMO
using_demo = not EXPORT_PATH
df = pd.DataFrame([{k: r.get(k) for k in ["index","category","n_tokens",*METRICS]} for r in records])
print(f"Loaded {len(df)} records" + ("  [DEMO — smoke test only]" if using_demo else ""))
df

## Part 1 — Is stress redundant with what you already log?

Two cheap tests on the three scalars you already export:

1. **Rank correlation** — does stress move monotonically with density or KL?
2. **The decisive one** — can a linear combination of density + KL *reconstruct* stress? If $R^2\approx 1$, stress adds no information beyond them (it's a restatement). If $R^2$ is low, stress carries independent variance you then get to characterize.

Rank (Spearman) correlation is used so a monotone-but-curved relationship still shows up.

In [ ]:
def rankdata(a):
    a = np.asarray(a, float)
    order = a.argsort(); ranks = np.empty(len(a)); ranks[order] = np.arange(len(a))
    vals, inv, counts = np.unique(a, return_inverse=True, return_counts=True)
    sums = np.zeros(len(counts)); np.add.at(sums, inv, ranks)
    return (sums / counts)[inv] + 1                      # average-tie ranks, 1-based

def spearman(x, y):
    return np.corrcoef(rankdata(x), rankdata(y))[0, 1]

print("Spearman rank correlations:")
for i, a in enumerate(METRICS):
    for b in METRICS[i+1:]:
        print(f"  {a:14s} vs {b:14s}  rho = {spearman(df[a], df[b]):+.3f}")

In [ ]:
# THE FIGURE — pairwise scatter, colored by category
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
pairs = [("density_mean","stress_score"), ("kl_divergence","stress_score"), ("density_mean","kl_divergence")]
cats = list(df.category.dropna().unique())
colors = {c: plt.cm.tab10(i % 10) for i, c in enumerate(cats)}
for ax, (xk, yk) in zip(axes, pairs):
    for c in cats:
        s = df[df.category == c]
        ax.scatter(s[xk], s[yk], label=c, color=colors[c], s=60, alpha=0.85, edgecolor="k", linewidth=0.4)
    ax.set_xlabel(xk); ax.set_ylabel(yk)
axes[0].legend(fontsize=7, loc="best")
fig.suptitle("Is stress an independent quantity?" + ("   [DEMO]" if using_demo else ""))
fig.tight_layout(); plt.show()

In [ ]:
# THE DECISIVE TEST — reconstruct stress from density + KL
X = np.column_stack([df.density_mean.values, df.kl_divergence.values, np.ones(len(df))])
y = df.stress_score.values.astype(float)
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ beta
r2 = 1 - resid.var() / y.var() if y.var() > 0 else float("nan")

print(f"stress  ~  a*density + b*kl + c")
print(f"  a (density) = {beta[0]:+.4f}")
print(f"  b (kl)      = {beta[1]:+.4f}")
print(f"  R^2         = {r2:.3f}")
print(f"  residual std = {resid.std():.4f}   (stress std = {y.std():.4f})")

p = 3
n_eff = len(np.unique(np.column_stack([df.density_mean, df.kl_divergence]), axis=0))
if n_eff <= p + 2:
    print(f"\n[!] Only {n_eff} DISTINCT records for {p} parameters — R^2 is overfit and MEANINGLESS.")
    print("    Point EXPORT_PATH at your full export (want dozens+). This is the smoke test.")
elif r2 > 0.9:
    print("\n=> High R^2: stress is largely a restatement of density+KL. Report that honestly.")
else:
    print(f"\n=> {(1-r2)*100:.0f}% of stress variance is NOT explained by density+KL.")
    print("   Stress carries independent signal — characterize what it tracks (Part 2).")

## How to read the result (decide, don't narrate)

- **High $R^2$ / high $|\rho|$ with density** → stress is density in different units. That is a clean, honest, *publishable* finding: you've shown two of your channels are one channel. Collapse them and move on.
- **Low $R^2$** → stress carries variance the named metrics don't. Now you have the interesting job: characterize *what* the residual tracks. That residual is the actual novelty claim, and it's now a measured quantity, not a vibe.
- **Sign of $\rho$ with density is negative** (as in the demo) → note it: stress and density may move *oppositely*, which would mean they're not the same thing pointed the same way. Worth understanding before you claim either.

Whatever the number, you can now write the sentence you said you couldn't: *"stress is $\frac{1}{L}\sum_{\ell,p}\lVert h\,\Delta W_p^\top\rVert/\lVert\Delta W_p\rVert_F$, and empirically it is / isn't reducible to the metrics I already log."* That sentence is a research note.

## Part 2 — To fully close it, add three columns

The export gives you stress, KL, density. To rule out the remaining confounds you need three quantities the export doesn't yet carry. Each is a small addition in `analyzer.py` next to `_extract_stress_score`:

1. **Mean next-token entropy per record** — rules out "stress tracks uncertainty."
2. **Activation norm $\lVert h_t\rVert$ (mean over tokens, same layers)** — rules out the units confound: because $\Delta W_p$ is in both numerator and denominator, stress scales like $\lVert h\rVert$. If stress $\approx c\lVert h\rVert$, it's an activation-magnitude meter, not an alignment-subspace meter. **This is the most important control and the one your current data can't run.**
3. **Per-token stress trace** (`per_token_stress`, already computed — just export it) — lets you ask the sharper question later: does stress *localize*, like the density-leverage test, or is it flat?

Then re-run the decisive test with `density, kl, entropy, h_norm` as predictors. If stress survives *that* regression with independent variance, you have a genuinely novel scalar and a real reason to name it. If it collapses onto $\lVert h\rVert$, you've learned that too — cheaply, before building anything else on it.